# pv per strain vs pv per mb scatter

built from `master_strains.csv` so pv counts reflect the filter flags.
toggle `SHOW_FILTERED` to switch between filtered and unfiltered views.

genome lengths come from `pv_per_fasta_with_lengths.csv` since they are
measured from the fasta files and not available in the phasomeit extraction.


## 1. settings


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from pathlib import Path
import sys

# -- Paths --------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd().parents[0]
STANDARD_DIR      = PROJECT_ROOT / "Standards"

if str(STANDARD_DIR) not in sys.path:
    sys.path.insert(0, str(STANDARD_DIR))

from colors import color_map, genus_order

MASTER_STRAINS    = PROJECT_ROOT / "Results" / "Tables" / "master_strains.csv"
PV_PER_MB_PATH    = PROJECT_ROOT / "Results" / "Tables" / "pv_per_fasta_with_lengths.csv"  # genome lengths only
FIG_DIR           = PROJECT_ROOT / "Results" / "Figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# -- Filter toggle -------------------------------------------------------------
# True  -> use only rows where passes_all_filters == True
# False -> use all PV rows (original unfiltered behaviour)
SHOW_FILTERED     = True

# -- Output filenames (suffix added automatically based on SHOW_FILTERED) ------
SUFFIX            = "_filtered" if SHOW_FILTERED else "_unfiltered"
FIG_NAME          = f"scatter_pv_per_strain_vs_pv_per_mb{SUFFIX}.png"
FIG_NAME_LEG      = f"scatter_pv_per_strain_vs_pv_per_mb_legend{SUFFIX}.png"

# -- Marker styling ------------------------------------------------------------
ARCH_MARKER       = "o"
BACT_MARKER       = "D"
MARKER_SIZE       = 180
MARKER_ALPHA      = 0.85
BACT_COLOR        = "#aaaaaa"
EDGE_COLOR        = "black"
EDGE_WIDTH        = 0.9

# -- Bacteria label offsets (x, y) in data units -- adjust as needed ----------
BACT_LABEL_OFFSETS = {
    "Brucella":      (35,  -33),
    "Campylobacter": (-260, -20),
    "Escherichia":   (35,  -20),
}
BACT_LABEL_FONTSIZE = 12

# -- Figure --------------------------------------------------------------------
FIG_SIZE          = (12, 7)
TITLE             = f"PPV per Strain vs PPV per Mb by Genus{'' if SHOW_FILTERED else '  [unfiltered]'}"
TITLE_FONTSIZE    = 22
TITLE_FONTWEIGHT  = "bold"
XLAB              = "PPV per Strain"
YLAB              = "Mean PPV per Mb"
GRID_ALPHA        = 0.35
GRID_LINESTYLE    = "--"
SAVE_DPI          = 300

# -- Legend --------------------------------------------------------------------
LEG_COLUMNS       = 6
LEG_FONTSIZE      = 13
LEG_TITLE_FONTSIZE = 16
LEG_MARKER_SIZE   = 12

# -- Archaea colour map (from shared Standards/colors.py) ----------------------
ARCH_COLOR_MAP    = color_map.copy()
ARCH_LEGEND_ORDER = list(genus_order)

plt.rcParams["font.family"]     = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial"]

print(f"SHOW_FILTERED = {SHOW_FILTERED}")
print(f"Output files  : {FIG_NAME}  |  {FIG_NAME_LEG}")

## 2. load data and compute per-strain pv counts


In [ ]:
ms       = pd.read_csv(MASTER_STRAINS, low_memory=False)
pv_fasta = pd.read_csv(PV_PER_MB_PATH, low_memory=False)

print(f"master_strains : {len(ms):,} rows")
print(f"pv_fasta       : {len(pv_fasta):,} rows")
print(f"pv_fasta cols  : {pv_fasta.columns.tolist()}")

# ── Subset to PV rows only (intragenic or upstream — these are the PV calls) ──
pv_rows = ms[ms["pv_class"].isin(["intragenic", "upstream"])].copy()

if SHOW_FILTERED:
    pv_rows = pv_rows[pv_rows["passes_all_filters"] == True]

# ── Count unique PV gene groups per strain ────────────────────────────────────
# One group can appear multiple times in a strain file if it has multiple tracts;
# count distinct group_nums to match PhasomeIt's group-level reporting.
pv_counts = (
    pv_rows
    .groupby(["domain", "genus", "strain_accession"])["group_num"]
    .nunique()
    .reset_index()
    .rename(columns={"group_num": "PV_count"})
)

# ── Join genome lengths ───────────────────────────────────────────────────────
# Identify the strain ID column in pv_fasta (may be fasta_name or sample_id)
strain_col = next(
    (c for c in pv_fasta.columns if "fasta" in c.lower() or "sample" in c.lower() or "accession" in c.lower()),
    pv_fasta.columns[0]
)
genome_len_col = next(
    (c for c in pv_fasta.columns if "length" in c.lower() or "genome" in c.lower()),
    None
)
print(f"\nUsing pv_fasta strain column   : '{strain_col}'")
print(f"Using pv_fasta genome length   : '{genome_len_col}'")

genome_lengths = pv_fasta[[strain_col, genome_len_col]].rename(
    columns={strain_col: "strain_accession", genome_len_col: "genome_length_bp"}
).drop_duplicates(subset="strain_accession")

pv_counts = pv_counts.merge(genome_lengths, on="strain_accession", how="left")

missing_lengths = pv_counts["genome_length_bp"].isna().sum()
if missing_lengths > 0:
    print(f"\n[WARN] {missing_lengths} strains have no genome length — they will be excluded from PV_per_Mb.")

# ── Compute PV per Mb ─────────────────────────────────────────────────────────
pv_counts["PV_per_Mb"] = pv_counts["PV_count"] / (pv_counts["genome_length_bp"] / 1_000_000)

print(f"\nPer-strain rows with PV_per_Mb : {pv_counts['PV_per_Mb'].notna().sum():,}")
display(pv_counts.head())

## 3. aggregate to genus level


In [ ]:
genus_means = (
    pv_counts
    .groupby(["domain", "genus"])[["PV_per_Mb", "PV_count"]]
    .mean()
    .reset_index()
    .rename(columns={"PV_count": "pv_per_strain", "PV_per_Mb": "pv_per_mb"})
)

arch = genus_means[genus_means["domain"] == "Archaea"]
bact = genus_means[genus_means["domain"] == "Bacteria"]

print(f"Archaea genera : {sorted(arch['genus'].tolist())}")
print(f"Bacteria genera: {sorted(bact['genus'].tolist())}")
print()
print(genus_means.sort_values("pv_per_strain", ascending=False).to_string(index=False))

## 4. figure 1 scatter


In [ ]:
fig, ax = plt.subplots(figsize=FIG_SIZE)

# -- Archaea circles -----------------------------------------------------------
for _, row in arch.iterrows():
    g     = row["genus"]
    color = ARCH_COLOR_MAP.get(g, "#888888")
    ax.scatter(
        row["pv_per_strain"], row["pv_per_mb"],
        c=color, s=MARKER_SIZE, marker=ARCH_MARKER,
        edgecolors=EDGE_COLOR, linewidths=EDGE_WIDTH,
        alpha=MARKER_ALPHA, zorder=3
    )

# -- Bacteria diamonds ---------------------------------------------------------
for _, row in bact.iterrows():
    g = row["genus"]
    ax.scatter(
        row["pv_per_strain"], row["pv_per_mb"],
        c=BACT_COLOR, s=MARKER_SIZE, marker=BACT_MARKER,
        edgecolors=EDGE_COLOR, linewidths=EDGE_WIDTH,
        alpha=MARKER_ALPHA, zorder=3
    )
    ox, oy = BACT_LABEL_OFFSETS.get(g, (20, 15))
    ax.annotate(
        g,
        xy=(row["pv_per_strain"], row["pv_per_mb"]),
        xytext=(row["pv_per_strain"] + ox, row["pv_per_mb"] + oy),
        fontsize=BACT_LABEL_FONTSIZE,
        va="bottom"
    )

# -- Axes ----------------------------------------------------------------------
ax.set_xlabel(XLAB,  fontsize=14, labelpad=10, fontweight="bold")
ax.set_ylabel(YLAB,  fontsize=14, labelpad=10, fontweight="bold")

ax.grid(True, linestyle=GRID_LINESTYLE, alpha=GRID_ALPHA)
ax.set_axisbelow(True)

# -- In-plot shape legend ------------------------------------------------------
shape_handles = [
    mlines.Line2D([], [], color="#555555", marker=ARCH_MARKER, linestyle="None",
                  markersize=12, markeredgecolor=EDGE_COLOR, markeredgewidth=0.8,
                  label="Archaea Genera"),
    mlines.Line2D([], [], color=BACT_COLOR, marker=BACT_MARKER, linestyle="None",
                  markersize=12, markeredgecolor=EDGE_COLOR, markeredgewidth=0.8,
                  label="Reference Bacteria"),
]
ax.legend(handles=shape_handles, fontsize=11, loc="upper left", frameon=True)

plt.tight_layout()
fig_path = FIG_DIR / FIG_NAME
plt.savefig(fig_path, dpi=SAVE_DPI, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

#view table of plotted points with genus, pv_per_strain, pv_per_mb
display(genus_means[["domain", "genus", "pv_per_strain", "pv_per_mb"]].sort_values("pv_per_strain", ascending=False).reset_index(drop=True))


In [ ]:
# Figure 1 inset export -- log y-scale, x-range 0-300, y-range up to 200, no axis labels
FIG_NAME_INSET = f"scatter_pv_per_strain_vs_pv_per_mb_inset_0_300_log{SUFFIX}.png"

# Log y-axis cannot include 0 exactly; use a small positive lower bound.
X_MIN, X_MAX = 0, 300
Y_MIN_LOG, Y_MAX = 10, 200

# Keep only points in the inset window so Brucella + in-range Archaea are explicitly included.
arch_inset = arch[
    (arch["pv_per_strain"] >= X_MIN) & (arch["pv_per_strain"] <= X_MAX) &
    (arch["pv_per_mb"] >= Y_MIN_LOG) & (arch["pv_per_mb"] <= Y_MAX)
].copy()

bact_inset = bact[
    (bact["pv_per_strain"] >= X_MIN) & (bact["pv_per_strain"] <= X_MAX) &
    (bact["pv_per_mb"] >= Y_MIN_LOG) & (bact["pv_per_mb"] <= Y_MAX)
].copy()

fig_inset, ax_inset = plt.subplots(figsize=(4, 3))

# Archaea in range
for _, row in arch_inset.iterrows():
    g = row["genus"]
    color = ARCH_COLOR_MAP.get(g, "#888888")
    ax_inset.scatter(
        row["pv_per_strain"], row["pv_per_mb"],
        c=color, s=MARKER_SIZE * 2.8, marker=ARCH_MARKER,
        edgecolors=EDGE_COLOR, linewidths=EDGE_WIDTH,
        alpha=MARKER_ALPHA, zorder=3
    )

# Bacteria in range (includes Brucella if present in window)
for _, row in bact_inset.iterrows():
    ax_inset.scatter(
        row["pv_per_strain"], row["pv_per_mb"],
        c=BACT_COLOR, s=MARKER_SIZE * 2.8, marker=BACT_MARKER,
        edgecolors=EDGE_COLOR, linewidths=EDGE_WIDTH,
        alpha=MARKER_ALPHA, zorder=3
    )

ax_inset.set_yscale("log")
ax_inset.set_xlim(X_MIN, X_MAX)
ax_inset.set_ylim(Y_MIN_LOG, Y_MAX)

# Inset style: no axis labels or tick labels
ax_inset.set_xlabel("")
ax_inset.set_ylabel("")
ax_inset.set_xticklabels([])
ax_inset.set_yticklabels([])

ax_inset.grid(True, linestyle=GRID_LINESTYLE, alpha=GRID_ALPHA)
ax_inset.set_axisbelow(True)

plt.tight_layout()
fig_path_inset = FIG_DIR / FIG_NAME_INSET
plt.savefig(fig_path_inset, dpi=SAVE_DPI, bbox_inches="tight", transparent=True)
plt.show()
print(f"Saved: {fig_path_inset}")
print(f"In inset: {len(arch_inset)} Archaea genera, {len(bact_inset)} Bacteria genera")
if "Brucella" in bact_inset["genus"].values:
    print("Brucella is included in inset range.")
else:
    print("Brucella is NOT in inset range.")

## 5. figure 2 standalone legend panel


In [ ]:
colour_handles = [
    mlines.Line2D([], [], color=ARCH_COLOR_MAP[g], marker=ARCH_MARKER,
                  linestyle="None", markersize=LEG_MARKER_SIZE,
                  markeredgecolor=EDGE_COLOR, markeredgewidth=0.6, label=g)
    for g in ARCH_LEGEND_ORDER if g in arch["genus"].values
]

n_handles  = len(colour_handles)
n_rows     = int(np.ceil(n_handles / LEG_COLUMNS))
leg_height = max(0.6, n_rows * 0.28 + 0.5)

fig_leg, ax_leg = plt.subplots(figsize=(FIG_SIZE[0], leg_height))
ax_leg.axis("off")

ax_leg.legend(
    handles=colour_handles,
    ncol=LEG_COLUMNS,
    loc="center",
    fontsize=LEG_FONTSIZE,
    title="Genera",
    title_fontsize=LEG_TITLE_FONTSIZE,
    frameon=True,
    borderpad=0.8,
    columnspacing=1.2,
    handletextpad=0.5,
)

plt.tight_layout()
fig_path_leg = FIG_DIR / FIG_NAME_LEG
plt.savefig(fig_path_leg, dpi=SAVE_DPI, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path_leg}")